# last_test_knn — KNN HPO + 다양성 axes (~18시간)

**목적**: 11-base stacking plateau (val=0.005701) 깰 수 있는 신규 다양성 base를 KNN(거리 기반 inductive bias) 아키텍처에서 탐색.

**탐색 axes (6축, Optuna TPE)**:
- `top_k_features` ∈ {30, 50, 100, 200} — gain importance 상위 K개 feature subset
- `n_neighbors` ∈ int log [5, 500] — 이웃 개수 K
- `weights` ∈ {uniform, distance} — 이웃 가중 방식
- `metric` ∈ {euclidean, manhattan} — 거리 함수
- `scaling` ∈ {standard, robust, quantile, hybrid} — 스케일러 종류 (HybridScaler 포함)
- `target_transform` ∈ {log1p, none} — zero-inflated 대응

**알고리즘**: Optuna TPE (n_startup_trials=25). KNN trial당 ~5~12분.

**예산**: 18시간 timeout. ~120~180 trials 예상.

**산출**: SQLite의 user_attrs에 모든 trial 메타 누적, val_rmse best 1세트만 `best_*.csv`로 덮어쓰기.

**게이트 (사후 검사)**: 단독 val < 0.006280 (1.10배) AND max_corr_with_11base < 0.97.

**의존성 (다른 PC 이식 시)**: `utils.config`, `utils.data`, `final.modules.preprocess` (전처리)만 필요.
HybridScaler·KNN 학습 로직·헬퍼는 모두 노트북 인라인.

**실행 방법**: 이 노트북만 다른 PC로 옮겨서 실행 (Colab/Local 자동 부트스트랩).

## 1. 환경 + import (self-contained)

In [ ]:
import os, sys, json, time, math

GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/final/modules/preprocess.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID 비어있음'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess

import optuna
from sklearn.model_selection import KFold
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler, RobustScaler, QuantileTransformer, PowerTransformer

optuna.logging.set_verbosity(optuna.logging.WARNING)

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'SEED = {SEED}')

## 2. 설정

In [ ]:
USER       = 'jh'
N_FOLDS    = 5
CLIP_Y_EXTREME = True

TIMEOUT_HOURS = 18.0
TIMEOUT_SEC   = int(TIMEOUT_HOURS * 3600)
N_TRIALS_CAP  = 300
N_STARTUP     = 25

OUT_DIR = os.path.join(OUTPUT_DIR, 'last_test_knn')
os.makedirs(OUT_DIR, exist_ok=True)

EXP_ID = 'last-test-knn-hpo'
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')

BEST_SINGLE_VAL = 0.005709
# KNN 단독 성능은 LGBM 대비 떨어질 가능성 높음 → 게이트 1을 1.10배로 완화 (다양성 후보 발굴 우선)
THR_SINGLE_VAL  = BEST_SINGLE_VAL * 1.10
THR_CORR_VS_11  = 0.97

# n_jobs: 솔로 실행 시 -1, 다른 노트북과 병행 시 코어 수의 절반
KNN_N_JOBS = -1

print(f'EXP_ID={EXP_ID}')
print(f'TIMEOUT={TIMEOUT_HOURS:.1f}h ({TIMEOUT_SEC}s) | N_TRIALS_CAP={N_TRIALS_CAP} | N_STARTUP={N_STARTUP}')
print(f'OUT_DIR={OUT_DIR}')
print(f'게이트: single_val<{THR_SINGLE_VAL:.6f}, max_corr_vs_11<{THR_CORR_VS_11}')
print(f'KNN_N_JOBS={KNN_N_JOBS}')

## 3. 데이터 로드 + 전처리 (1회)

11-base와 동일한 전처리 PARAMS로 전처리. 다른 PC 이식 시 `final.modules.preprocess`만 있으면 동작.

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)

ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = int((y_raw >= 1.0).sum())
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip ({n_clipped}개)')

y_train_unit = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
y_val_unit   = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
y_test_unit  = ys_input['test'].set_index(KEY_COL)[TARGET_COL]

PARAMS = {
    'missing_threshold':          0.5,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.25,
    'spatial_max_dist':           5.0,
    'post_impute_corr_threshold': 0.99,
    'post_impute_corr_keep_by':   'std',
}
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train_die = pp['xs_train']
xs_val_die   = pp['xs_val']
xs_test_die  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

X_train_die = xs_train_die[feat_cols_clean].values.astype(np.float32)
X_val_die   = xs_val_die[feat_cols_clean].values.astype(np.float32)
X_test_die  = xs_test_die[feat_cols_clean].values.astype(np.float32)
uid_train_die = xs_train_die[KEY_COL].values
uid_val_die   = xs_val_die[KEY_COL].values
uid_test_die  = xs_test_die[KEY_COL].values

y_train_die_broadcast = pd.Series(uid_train_die).map(y_train_unit).values.astype(np.float32)
assert not np.isnan(y_train_die_broadcast).any(), 'unmapped train die y'

unit_ids_train_unique = y_train_unit.index.values
kf_global = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = list(kf_global.split(unit_ids_train_unique))

n_train_die = len(X_train_die)
n_val_die   = len(X_val_die)
n_test_die  = len(X_test_die)

print(f'\n[cleaning] feat_cols_clean={len(feat_cols_clean)}')
print(f'  X_train_die: {X_train_die.shape}, val: {X_val_die.shape}, test: {X_test_die.shape}')
print(f'  unit train={len(y_train_unit):,}, val={len(y_val_unit):,}, test={len(y_test_unit):,}')
print(f'  fold: {N_FOLDS}-fold unit-level shuffle SEED={SEED}')

## 4. HybridScaler 인라인 + 스케일러 factory

`final/modules/scaling.py` v4 (binary passthrough + |skew|>threshold→Quantile + Power) 의 사본을 인라인으로 둠.
다른 PC 환경의 모듈 버전 차이를 차단하기 위함.

In [ ]:
class HybridScaler:
    """
    Skew 기반 하이브리드 스케일러 (sklearn fit/transform 패턴)
      1) Binary passthrough : nunique ≤ 2 (변환 없음)
      2) Quantile 변환      : |skew| > threshold (rank 기반, heavy-tail 평탄화)
      3) Power(Yeo-Johnson) : 나머지 (standardize=True, mean=0/std=1)
    """
    def __init__(self, skew_threshold=10.0, n_quantiles=1000,
                 quantile_output='normal', random_state=42,
                 binary_passthrough=True, verbose=False):
        self.skew_threshold = skew_threshold
        self.n_quantiles = n_quantiles
        self.quantile_output = quantile_output
        self.random_state = random_state
        self.binary_passthrough = binary_passthrough
        self.verbose = verbose

    def fit(self, X, feat_cols=None):
        if feat_cols is None:
            feat_cols = list(X.columns)
        self.feat_cols_ = list(feat_cols)
        if self.binary_passthrough:
            nuniq = X[self.feat_cols_].nunique()
            self.binary_cols_ = nuniq[nuniq <= 2].index.tolist()
        else:
            self.binary_cols_ = []
        remaining = [c for c in self.feat_cols_ if c not in set(self.binary_cols_)]
        if remaining:
            skew_vals = X[remaining].skew().abs()
        else:
            skew_vals = pd.Series(dtype=float)
        self.quantile_cols_ = skew_vals[skew_vals > self.skew_threshold].index.tolist()
        self.power_cols_ = [c for c in remaining if c not in set(self.quantile_cols_)]
        self.qt_ = None
        if self.quantile_cols_:
            n_q = min(self.n_quantiles, len(X))
            self.qt_ = QuantileTransformer(
                n_quantiles=n_q,
                output_distribution=self.quantile_output,
                subsample=int(1e6),
                random_state=self.random_state,
            )
            self.qt_.fit(X[self.quantile_cols_])
        self.pt_ = None
        if self.power_cols_:
            self.pt_ = PowerTransformer(method='yeo-johnson', standardize=True)
            self.pt_.fit(X[self.power_cols_])
        if self.verbose:
            print(f'[HybridScaler] binary={len(self.binary_cols_)}, '
                  f'quantile={len(self.quantile_cols_)}, power={len(self.power_cols_)}')
        return self

    def transform(self, X, inplace=True):
        if not inplace:
            X = X.copy()
        if self.quantile_cols_ and self.qt_ is not None:
            arr = self.qt_.transform(X[self.quantile_cols_])
            in_dtype = X[self.quantile_cols_].dtypes.iloc[0]
            if in_dtype == np.float32:
                arr = arr.astype('float32', copy=False)
            X[self.quantile_cols_] = arr
        if self.power_cols_ and self.pt_ is not None:
            arr = self.pt_.transform(X[self.power_cols_])
            in_dtype = X[self.power_cols_].dtypes.iloc[0]
            if in_dtype == np.float32:
                arr = arr.astype('float32', copy=False)
            X[self.power_cols_] = arr
        return X


def fit_scaler_on_train_np(name, Xtr_np, feat_names):
    """Train fold에서 fit. .transform(np.ndarray) → np.ndarray 인터페이스로 반환."""
    if name == 'standard':
        return StandardScaler().fit(Xtr_np)
    if name == 'robust':
        return RobustScaler().fit(Xtr_np)
    if name == 'quantile':
        return QuantileTransformer(
            n_quantiles=min(1000, len(Xtr_np)),
            output_distribution='normal',
            subsample=int(1e6),
            random_state=SEED,
        ).fit(Xtr_np)
    if name == 'hybrid':
        df = pd.DataFrame(Xtr_np, columns=feat_names)
        sc = HybridScaler(skew_threshold=10.0).fit(df, feat_names)
        names = list(feat_names)
        class _NPHybridWrap:
            def transform(self, X_np):
                d = pd.DataFrame(X_np, columns=names)
                sc.transform(d, inplace=True)
                return d[names].values
        return _NPHybridWrap()
    raise ValueError(name)


print('HybridScaler 인라인 정의 + 스케일러 factory 정의 완료')

## 5. Feature gain importance 캐싱 (top-K base list)

`reg_only/lgbm/fold_models.pkl`의 5-fold 평균 gain importance 기반 정렬. trial마다 top_k 슬라이스.

In [ ]:
import pickle

with open(os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm', 'fold_models.pkl'), 'rb') as f:
    fm_data = pickle.load(f)

if isinstance(fm_data, dict):
    fold_models_list = fm_data.get('fold_models', None)
    lgbm_feat_names  = fm_data.get('feature_names', None)
else:
    fold_models_list = list(fm_data)
    lgbm_feat_names  = None

assert fold_models_list is not None and len(fold_models_list) > 0, 'fold_models.pkl 비어있음'

def _recover_feat_names(models):
    m0 = models[0]
    if hasattr(m0, 'booster_'):
        return list(m0.booster_.feature_name())
    if hasattr(m0, 'feature_name_'):
        return list(m0.feature_name_)
    raise RuntimeError('cached LGBM model에서 feature_name 복구 실패')

if lgbm_feat_names is None or not hasattr(lgbm_feat_names, '__iter__'):
    print('[fold_models.pkl] feature_names=None → booster에서 복구')
    lgbm_feat_names = _recover_feat_names(fold_models_list)
else:
    lgbm_feat_names = list(lgbm_feat_names)

print(f'fold_models: {len(fold_models_list)}개 | feature_names: {len(lgbm_feat_names)}개')
print(f'  lgbm_feat 첫 5: {lgbm_feat_names[:5]}')
print(f'  feat_cols_clean 첫 5: {feat_cols_clean[:5]}')

gains = []
for m in fold_models_list:
    if hasattr(m, 'booster_'):
        gains.append(m.booster_.feature_importance(importance_type='gain'))
    elif hasattr(m, 'feature_importances_'):
        gains.append(m.feature_importances_)
mean_gain = np.mean(gains, axis=0)
assert len(mean_gain) == len(lgbm_feat_names), \
    f'gain 길이({len(mean_gain)}) != feat_names({len(lgbm_feat_names)})'

common_feats = [f for f in lgbm_feat_names if f in feat_cols_clean]
print(f'  교집합 (lgbm_feat_names ∩ feat_cols_clean): {len(common_feats)}개')

MIN_REQUIRED = 200  # max top_k

if len(common_feats) >= MIN_REQUIRED:
    # 정상 경로: gain importance 기반 정렬
    gain_dict = {f: g for f, g in zip(lgbm_feat_names, mean_gain)}
    common_gains = np.array([gain_dict[f] for f in common_feats])
    order = np.argsort(-common_gains)
    TOP_FEATS_LIST = [common_feats[i] for i in order]
    TOP_FEATS_IDX_IN_CLEAN = [feat_cols_clean.index(f) for f in TOP_FEATS_LIST]
    print(f'  → gain importance 정렬 사용 ({len(TOP_FEATS_LIST)}개)')
else:
    # Fallback: feat_cols_clean의 |target 상관계수| 내림차순 사용
    print(f'[WARN] 교집합 부족 ({len(common_feats)} < {MIN_REQUIRED}) → '
          f'feat_cols_clean 전체({len(feat_cols_clean)}개)를 |target 상관| 순으로 fallback')
    y_die = y_train_die_broadcast.astype(np.float64)
    target_corrs = []
    for j in range(X_train_die.shape[1]):
        col = X_train_die[:, j].astype(np.float64)
        if col.std() < 1e-12:
            target_corrs.append(0.0)
        else:
            c = float(np.corrcoef(col, y_die)[0, 1])
            target_corrs.append(0.0 if not np.isfinite(c) else abs(c))
    target_corrs = np.array(target_corrs)
    order = np.argsort(-target_corrs)
    TOP_FEATS_LIST = [feat_cols_clean[i] for i in order]
    TOP_FEATS_IDX_IN_CLEAN = list(order)
    print(f'  → fallback 정렬 사용 ({len(TOP_FEATS_LIST)}개, 상위 5: {TOP_FEATS_LIST[:5]})')

assert len(TOP_FEATS_LIST) >= MIN_REQUIRED, \
    f'TOP_FEATS_LIST 부족: {len(TOP_FEATS_LIST)} (need >= {MIN_REQUIRED})'

print(f'top-K base list 캐싱: 총 {len(TOP_FEATS_LIST)}개')
print(f'  top-5: {TOP_FEATS_LIST[:5]}')

## 6. 11-base OOF 잔차 로드 (게이트 corr 계산용)

In [ ]:
BASE_OOF_PATHS = {
    'zit_only':                 os.path.join(OUTPUT_DIR, 'final', 'zit_only',      'oof_unit.csv'),
    'bag_zit_combined_best':    os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best',    'oof_unit.csv'),
    'bag_zit_hpo':              os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_hpo',              'oof_unit.csv'),
    'bag_zit_combined_best_xy': os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_combined_best_xy', 'oof_unit.csv'),
    'bag_zit_pp_hpo':           os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_pp_hpo',           'oof_unit.csv'),
    'bag_zit_fixed_ge':         os.path.join(OUTPUT_DIR, '_temp', 'bag_zit_fixed_ge',         'oof_unit.csv'),
    'reg__catboost':            os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'catboost', 'oof_unit.csv'),
    'reg__lgbm':                os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'lgbm',     'oof_unit.csv'),
    'reg__et':                  os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'et',       'oof_unit.csv'),
    'reg__enet':                os.path.join(OUTPUT_DIR, 'final', 'reg_only', 'enet',     'oof_unit.csv'),
}

# 존재하는 파일만 로드 (다른 PC에서 일부 base가 동기화 안 돼 있어도 학습은 계속)
existing_paths = {k: p for k, p in BASE_OOF_PATHS.items() if os.path.exists(p)}
missing = [k for k in BASE_OOF_PATHS if k not in existing_paths]
if missing:
    print(f'[11-base OOF 일부 누락 — 학습은 계속, 게이트는 가능한 것만으로 계산]')
    print(f'  사용 가능: {len(existing_paths)}/{len(BASE_OOF_PATHS)}')
    for k in missing:
        print(f'  ✗ {k}  (skip)')

def _load_base_pred(path, y_series):
    df = pd.read_csv(path)
    return df.set_index(KEY_COL)['pred'].reindex(y_series.index).values

if existing_paths:
    base_oof_pred  = {k: _load_base_pred(p, y_train_unit) for k, p in existing_paths.items()}
    base_oof_resid = {k: y_train_unit.values - v for k, v in base_oof_pred.items()}
    print(f'[11-base OOF 잔차 로드] {len(base_oof_resid)}/{len(BASE_OOF_PATHS)}개')
else:
    base_oof_pred  = {}
    base_oof_resid = {}
    print(f'[WARN] 11-base 0개 — 게이트 corr 비활성화 (val_rmse만 보고 best 선택, 학습은 계속)')

## 7. 헬퍼 함수 (KNN train + best save + max_corr)

In [ ]:
def _mean_die_to_unit(pred_die, uid_die):
    unit_id = np.asarray(uid_die)
    unique_units, inverse = np.unique(unit_id, return_inverse=True)
    pred_sum = np.zeros(len(unique_units), dtype=np.float64)
    cnt      = np.zeros(len(unique_units), dtype=np.float64)
    np.add.at(pred_sum, inverse, pred_die)
    np.add.at(cnt,      inverse, 1.0)
    return pred_sum / cnt, unique_units


def _rmse_unit(pred_unit_arr, unique_units, y_unit_series):
    s = pd.Series(pred_unit_arr, index=unique_units).reindex(y_unit_series.index)
    return float(np.sqrt(np.mean((s.values - y_unit_series.values) ** 2)))


def _save_unit_csv(uids, pred, y_true, path):
    pd.DataFrame({KEY_COL: uids, 'pred': pred, TARGET_COL: y_true}).to_csv(path, index=False)


def _max_corr_with_11base(oof_unit_pred_aligned):
    # 11-base OOF가 없으면 게이트 비활성화 (corr=-1.0 sentinel — 학습 결과만 살림)
    if not base_oof_resid:
        return -1.0
    new_resid = y_train_unit.values - oof_unit_pred_aligned
    if np.std(new_resid) < 1e-12:
        return 1.0
    cs = []
    for r in base_oof_resid.values():
        c = float(np.corrcoef(new_resid, r)[0, 1])
        if not np.isfinite(c):
            c = 1.0
        cs.append(c)
    return max(cs)


BEST_VAL_RMSE = float('inf')

def _maybe_save_best(oof_s, val_s, test_s, val_rmse, meta_dict):
    """새 best (val_rmse 기준)이면 best_*.csv + best_meta.json 덮어쓰기."""
    global BEST_VAL_RMSE
    if val_rmse < BEST_VAL_RMSE:
        BEST_VAL_RMSE = val_rmse
        _save_unit_csv(y_train_unit.index.values, oof_s.values,  y_train_unit.values, os.path.join(OUT_DIR, 'best_oof_unit.csv'))
        _save_unit_csv(y_val_unit.index.values,   val_s.values,  y_val_unit.values,   os.path.join(OUT_DIR, 'best_val_unit.csv'))
        _save_unit_csv(y_test_unit.index.values,  test_s.values, y_test_unit.values,  os.path.join(OUT_DIR, 'best_test_unit.csv'))
        with open(os.path.join(OUT_DIR, 'best_meta.json'), 'w', encoding='utf-8') as f:
            json.dump(meta_dict, f, indent=2, ensure_ascii=False, default=str)
        return True
    return False


def train_knn_trial(top_k, n_neighbors, weights, metric, scaling_name, target_transform):
    """5-fold die-level KNN 학습 + die→unit 평균 집계 + 단독 RMSE/max_corr 반환."""
    feat_idx_top = TOP_FEATS_IDX_IN_CLEAN[:top_k]
    feat_names_top = TOP_FEATS_LIST[:top_k]
    Xtr_full = X_train_die[:, feat_idx_top]
    Xvl_full = X_val_die[:,   feat_idx_top]
    Xte_full = X_test_die[:,  feat_idx_top]

    oof_die  = np.full(n_train_die, np.nan, dtype=np.float64)
    val_die  = np.zeros(n_val_die,  dtype=np.float64)
    test_die = np.zeros(n_test_die, dtype=np.float64)

    t0 = time.time()
    for fold, (tr_uidx, vl_uidx) in enumerate(FOLDS):
        tu = unit_ids_train_unique[tr_uidx]
        vu = unit_ids_train_unique[vl_uidx]
        tr_mask = np.isin(uid_train_die, tu)
        vl_mask = np.isin(uid_train_die, vu)

        Xtr_f = Xtr_full[tr_mask].astype(np.float64, copy=False)
        Xvl_f = Xtr_full[vl_mask].astype(np.float64, copy=False)
        ytr_f = y_train_die_broadcast[tr_mask].astype(np.float64)

        sc = fit_scaler_on_train_np(scaling_name, Xtr_f, feat_names_top)
        Xtr_s = sc.transform(Xtr_f).astype(np.float64, copy=False)
        Xvl_s = sc.transform(Xvl_f).astype(np.float64, copy=False)
        Xext_vl_s = sc.transform(Xvl_full.astype(np.float64, copy=False)).astype(np.float64, copy=False)
        Xext_te_s = sc.transform(Xte_full.astype(np.float64, copy=False)).astype(np.float64, copy=False)

        if target_transform == 'log1p':
            ytr_fit = np.log1p(ytr_f)
        else:
            ytr_fit = ytr_f

        knn = KNeighborsRegressor(
            n_neighbors=n_neighbors, weights=weights, metric=metric,
            algorithm='auto', n_jobs=KNN_N_JOBS,
        )
        knn.fit(Xtr_s, ytr_fit)
        p_vl = knn.predict(Xvl_s)
        p_ev = knn.predict(Xext_vl_s)
        p_et = knn.predict(Xext_te_s)

        if target_transform == 'log1p':
            p_vl = np.clip(np.expm1(p_vl), 0.0, None)
            p_ev = np.clip(np.expm1(p_ev), 0.0, None)
            p_et = np.clip(np.expm1(p_et), 0.0, None)
        else:
            p_vl = np.clip(p_vl, 0.0, None)
            p_ev = np.clip(p_ev, 0.0, None)
            p_et = np.clip(p_et, 0.0, None)

        oof_die[vl_mask] = p_vl
        val_die  += p_ev / N_FOLDS
        test_die += p_et / N_FOLDS

    assert not np.isnan(oof_die).any()

    oof_u, oof_uids   = _mean_die_to_unit(oof_die,  uid_train_die)
    val_u, val_uids   = _mean_die_to_unit(val_die,  uid_val_die)
    test_u, test_uids = _mean_die_to_unit(test_die, uid_test_die)

    oof_rmse  = _rmse_unit(oof_u,  oof_uids,  y_train_unit)
    val_rmse  = _rmse_unit(val_u,  val_uids,  y_val_unit)
    test_rmse = _rmse_unit(test_u, test_uids, y_test_unit)

    oof_s  = pd.Series(oof_u,  index=oof_uids).reindex(y_train_unit.index)
    val_s  = pd.Series(val_u,  index=val_uids).reindex(y_val_unit.index)
    test_s = pd.Series(test_u, index=test_uids).reindex(y_test_unit.index)

    max_corr11 = _max_corr_with_11base(oof_s.values)

    return {
        'oof_rmse': oof_rmse, 'val_rmse': val_rmse, 'test_rmse': test_rmse,
        'max_corr_vs_11base': max_corr11,
        'oof_s': oof_s, 'val_s': val_s, 'test_s': test_s,
        'top_k': top_k,
        'elapsed': time.time() - t0,
    }


print('train_knn_trial 정의 완료')

## 8. Optuna HPO 실행 (TIMEOUT_SEC 자동 종료)

In [ ]:
def objective_knn(trial):
    top_k       = trial.suggest_categorical('top_k_features', [30, 50, 100, 200])
    n_neighbors = trial.suggest_int('n_neighbors', 5, 500, log=True)
    weights     = trial.suggest_categorical('weights', ['uniform', 'distance'])
    metric      = trial.suggest_categorical('metric',  ['euclidean', 'manhattan'])
    scaling     = trial.suggest_categorical('scaling', ['standard', 'robust', 'quantile', 'hybrid'])
    target_tt   = trial.suggest_categorical('target_transform', ['log1p', 'none'])

    r = train_knn_trial(top_k, n_neighbors, weights, metric, scaling, target_tt)

    trial.set_user_attr('val_rmse',           r['val_rmse'])
    trial.set_user_attr('test_rmse',          r['test_rmse'])
    trial.set_user_attr('max_corr_vs_11base', r['max_corr_vs_11base'])
    trial.set_user_attr('elapsed_s',          r['elapsed'])

    meta_dict = {
        'trial_number': trial.number,
        'oof_rmse': r['oof_rmse'], 'val_rmse': r['val_rmse'], 'test_rmse': r['test_rmse'],
        'max_corr_vs_11base': r['max_corr_vs_11base'],
        'top_k_features': top_k, 'n_neighbors': n_neighbors,
        'weights': weights, 'metric': metric,
        'scaling': scaling, 'target_transform': target_tt,
        'CLIP_Y_EXTREME': CLIP_Y_EXTREME,
        'feat_cols_clean_n': len(feat_cols_clean),
        'SEED': int(SEED),
        'elapsed_seconds': r['elapsed'],
    }
    is_new_best = _maybe_save_best(r['oof_s'], r['val_s'], r['test_s'], r['val_rmse'], meta_dict)
    best_mark = ' ★new_best' if is_new_best else ''

    p1 = '✅' if r['val_rmse'] < THR_SINGLE_VAL else '❌'
    p2 = '✅' if r['max_corr_vs_11base'] < THR_CORR_VS_11 else '❌'
    print(f'trial {trial.number:04d} | top={top_k:3d} K={n_neighbors:3d} w={weights[:4]:<4s} '
          f'm={metric[:3]} sc={scaling[:5]:<5s} tt={target_tt[:5]:<5s} '
          f'| val={r["val_rmse"]:.6f}{p1} corr={r["max_corr_vs_11base"]:.4f}{p2} '
          f'| {r["elapsed"]:.0f}s{best_mark}')

    return r['oof_rmse']


study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=True,
    sampler=optuna.samplers.TPESampler(seed=SEED, n_startup_trials=N_STARTUP),
)
study.set_user_attr('exp_memo', 'last_test_knn — KNN HPO (top_k × n_neighbors × weights × metric × scaling × target_transform)')
study.set_user_attr('timeout_hours', TIMEOUT_HOURS)

for t in study.trials:
    if t.state == optuna.trial.TrialState.COMPLETE:
        v = t.user_attrs.get('val_rmse', float('inf'))
        if v < BEST_VAL_RMSE:
            BEST_VAL_RMSE = v
print(f'\n현재 best_val_rmse (복구): {BEST_VAL_RMSE if BEST_VAL_RMSE != float("inf") else "없음"}')

print(f'\n=== Optuna HPO 시작 (timeout={TIMEOUT_HOURS}h, startup={N_STARTUP}, sampler=TPE) ===')
print(f'  탐색 axes: top_k(4) × n_neighbors(int log) × weights(2) × metric(2) × scaling(4) × target_transform(2)')
t_total = time.time()
try:
    study.optimize(objective_knn, n_trials=N_TRIALS_CAP, timeout=TIMEOUT_SEC)
except KeyboardInterrupt:
    print('\n[KeyboardInterrupt] HPO 중단 — 지금까지 trial은 SQLite에 저장됨')

print(f'\n[HPO 종료] {time.time()-t_total:.0f}s, completed trials: {len(study.trials)}')
print(f'best OOF RMSE = {study.best_value:.6f}')
print(f'best params:')
for k, v in study.best_trial.params.items():
    print(f'  {k:25s} = {v}')

## 9. 전체 trial summary + 게이트 통과 후보 출력

In [ ]:
rows = []
for t in study.trials:
    if t.state != optuna.trial.TrialState.COMPLETE:
        continue
    row = {
        'trial':       t.number,
        'oof_rmse':    t.value,
        'val_rmse':    t.user_attrs.get('val_rmse'),
        'test_rmse':   t.user_attrs.get('test_rmse'),
        'max_corr_11': t.user_attrs.get('max_corr_vs_11base'),
        'elapsed_s':   t.user_attrs.get('elapsed_s'),
        **{f'param_{k}': v for k, v in t.params.items()},
    }
    row['pass_1'] = (row['val_rmse'] is not None) and (row['val_rmse'] < THR_SINGLE_VAL)
    row['pass_2'] = (row['max_corr_11'] is not None) and (row['max_corr_11'] < THR_CORR_VS_11)
    row['pre_pass'] = row['pass_1'] and row['pass_2']
    rows.append(row)

if len(rows) == 0:
    print('=== 완료된 trial 없음 ===')
    n_fail = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.FAIL)
    n_run  = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.RUNNING)
    print(f'  FAIL: {n_fail} | RUNNING: {n_run} | total: {len(study.trials)}')
    summary_df = pd.DataFrame()
else:
    summary_df = pd.DataFrame(rows).sort_values('val_rmse')
    summary_df.to_csv(os.path.join(OUT_DIR, 'summary.csv'), index=False)

    print(f'=== 전체 complete trial: {len(summary_df)} ===')
    print(f'  게이트 1 (val<{THR_SINGLE_VAL:.6f}) 통과: {summary_df["pass_1"].sum()}')
    print(f'  게이트 2 (corr<{THR_CORR_VS_11}) 통과: {summary_df["pass_2"].sum()}')
    print(f'  둘 다 통과 (pre_pass): {summary_df["pre_pass"].sum()}')

    key_cols = ['trial','val_rmse','test_rmse','max_corr_11','pass_1','pass_2','pre_pass',
                'param_top_k_features','param_n_neighbors','param_weights','param_metric',
                'param_scaling','param_target_transform']
    display_cols = [c for c in key_cols if c in summary_df.columns]

    print(f'\n=== 상위 10 (val_rmse 기준) ===')
    print(summary_df.head(10)[display_cols].to_string(index=False))

    pass_df = summary_df[summary_df['pre_pass']]
    print(f'\n=== 게이트 통과 후보: {len(pass_df)}개 ===')
    if len(pass_df) > 0:
        print(pass_df.head(30)[display_cols].to_string(index=False))
        print(f'\n⚠ 통과 trial이 best_val_rmse 아닌 경우, 그 trial의 OOF는 SQLite엔 메타만 있고 CSV 없음.')
        print(f'   필요 시 해당 HP로 재실행해서 CSV 받기.')

print(f'\n=== best_*.csv 상태 (val_rmse={BEST_VAL_RMSE:.6f}) ===')
for fn in ['best_oof_unit.csv', 'best_val_unit.csv', 'best_test_unit.csv', 'best_meta.json']:
    p = os.path.join(OUT_DIR, fn)
    if os.path.exists(p):
        sz = os.path.getsize(p) / 1024
        print(f'  {fn:25s}  {sz:>8.1f} KB')
    else:
        print(f'  {fn:25s}  (없음)')

# Colab → 로컬 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', os.path.basename(OUT_DIR) + '_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'\n[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024/1024:.1f} MB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass